In [78]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error
import xgboost as xgb
import lightgbm as lgb
import warnings
warnings.filterwarnings("ignore")

In [79]:
df = pd.read_csv("tokyo_weather.csv")
df["Date"] = pd.to_datetime(df["Date"])
df = df.set_index("Date")

# Column we want to predict
TARGET = "Highest Temperature (°C)"

# Convert all columns to numeric values
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")




In [80]:

def create_features(data):
    """
    Create date-based features from the Date index.
    These help the model learn seasonal patterns.
    """
    data = data.copy()

    data["dayofweek"] = data.index.dayofweek
    data["month"] = data.index.month
    data["dayofyear"] = data.index.dayofyear
    data["dayofmonth"] = data.index.day
    data["weekofyear"] = data.index.isocalendar().week.astype(int)
    
    ##test features

    


    return data


def add_lags(data):
    """
    Create lag features.
    These tell the model what the temperature was on previous days.
    """
    data = data.copy()

    data["lag1_day"] = data[TARGET].shift(1)
    data["lag2_day"] = data[TARGET].shift(2)
    data["lag3_day"] = data[TARGET].shift(3)
    data["lag7_day"] = data[TARGET].shift(7)

    #test
    data["rolling_7"] = data[TARGET].shift(1).rolling(7).mean()
    data["rolling_3"] = data[TARGET].shift(1).rolling(3).mean()
    data["rolling_2"] = data[TARGET].shift(1).rolling(2).mean()
    
    data["diff_from_avg_7"] = data[TARGET].shift(1) - data["rolling_7"]

    return data


df = create_features(df)
df = add_lags(df)

# Remove rows where lag values are missing
needed_cols = [TARGET, "lag1_day", "lag2_day", "lag3_day", "lag7_day"]
df = df.dropna(subset=needed_cols)



In [81]:

final_test_data = df.iloc[-15:]

df = df.iloc[:-15]

ensembles_train_data = df[-200:]




#final data to train the stock models
df = df.iloc[:-200]



In [82]:
def train_xgb(X_train, y_train):
    model = xgb.XGBRegressor(
        n_estimators=500,
        learning_rate=0.03,
        max_depth=5,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42
    )
    model.fit(X_train, y_train)
    return model

In [83]:
def train_lgb(X_train, y_train):
    model = lgb.LGBMRegressor(
        n_estimators=500,
        learning_rate=0.03,
        max_depth=5,
        subsample=0.8,
        colsample_bytree=0.8,
        objective='regression',
        random_state=42
    )
    model.fit(X_train, y_train)
    return model

In [84]:
def train_cat(X_train, y_train):
    model = CatBoostRegressor(
        iterations=500,
        learning_rate=0.03,
        depth=6,
        loss_function='RMSE',
        random_seed=42,
        verbose=0
    )
    model.fit(X_train, y_train)
    return model

In [85]:
# Use every column except the target as input features for training
FEATURES = [col for col in df.columns if col != TARGET]


train_df = df[:-15]
test_df  = df[-15:]

X = train_df[FEATURES]
y = train_df[TARGET]

X_test = test_df[FEATURES]
y_test = test_df[TARGET]

base_models = [
    ("xgb", train_xgb),
    ("lgb", train_lgb),
    ("cat", train_cat)
]


In [90]:

def get_cv_validation_predictions(
    data,
    FEATURES,
    TARGET,
    models,
    n_splits=6,
    max_test_size=15
):
  

    test_size = min(max_test_size, len(data) // (n_splits + 1))

    tss = TimeSeriesSplit(
        n_splits=n_splits,
        test_size=test_size
    )

    all_fold_predictions = []

    for fold, (train_idx, val_idx) in enumerate(tss.split(data), start=1):

        train = data.iloc[train_idx]
        val = data.iloc[val_idx]

        X_train = train[FEATURES]
        y_train = train[TARGET]

        X_val = val[FEATURES]
        y_val = val[TARGET]

        fold_predictions = pd.DataFrame(index=val.index)

        fold_predictions["fold"] = fold
        fold_predictions["actual"] = y_val.values

        # Optional: keep the date if your dataframe has a Date column
        if "Date" in val.columns:
            fold_predictions["Date"] = val["Date"].values

        for name, train_func in models:

            model = train_func(X_train, y_train)

            preds = model.predict(X_val)

            fold_predictions[f"{name}_pred"] = preds

        all_fold_predictions.append(fold_predictions)

    validation_predictions = pd.concat(all_fold_predictions)

    return validation_predictions



validation_predictions = get_cv_validation_predictions(
    data=df,
    FEATURES=FEATURES,
    TARGET=TARGET,
    models=base_models,
    n_splits=20,
    max_test_size=15
)

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000250 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3756
[LightGBM] [Info] Number of data points in the train set: 1671, number of used features: 23
[LightGBM] [Info] Start training from score 21.262298
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive 

In [ ]:
validation_predictions


            fold  actual   xgb_pred   lgb_pred   cat_pred
Date                                                     
2023-01-29     1     8.8   8.010312   7.561102   8.405767
2023-01-30     1    10.0   9.761539   9.443845   9.904198
2023-01-31     1     9.0   9.799976  10.034868   9.763336
2023-02-01     1    13.1  13.394413  13.237518  12.905988
2023-02-02     1     9.2  11.071206  11.348787  10.828319
...          ...     ...        ...        ...        ...
2023-11-20    20    18.0  17.674612  17.581019  17.841578
2023-11-21    20    17.6  17.300936  17.102962  17.255010
2023-11-22    20    19.5  19.158421  19.008524  18.959855
2023-11-23    20    21.2  19.888123  19.638598  19.958051
2023-11-24    20    24.2  22.164513  21.963193  21.605652

[300 rows x 5 columns]


300

In [97]:
from sklearn.svm import SVR
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor


# --------------------------------------------------
# 2. Build ensemble training data
# --------------------------------------------------

ensemble_features = ["xgb_pred", "lgb_pred", "cat_pred"]

X_ensemble = validation_predictions[ensemble_features]
y_ensemble = validation_predictions["actual"]

# --------------------------------------------------
# 3. Train ensemble layer
# --------------------------------------------------

ensemble_model = XGBRegressor(
    n_estimators=100,
    learning_rate=0.03,
    max_depth=2,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42
)

ensemble_model.fit(X_ensemble, y_ensemble)
# --------------------------------------------------
# 4. Train base models on full df
# --------------------------------------------------

fitted_base_models = {}

X_full = df[FEATURES]
y_full = df[TARGET]

for name, train_func in base_models:
    fitted_base_models[name] = train_func(X_full, y_full)

# --------------------------------------------------
# 5. Predict using ensemble layer
# --------------------------------------------------

final_test = df[-15:]

X_final_test = final_test[FEATURES]
y_final_test = final_test[TARGET]

final_base_predictions = pd.DataFrame(index=final_test.index)

for name, model in fitted_base_models.items():
    final_base_predictions[f"{name}_pred"] = model.predict(X_final_test)

final_predictions = ensemble_model.predict(final_base_predictions)

# --------------------------------------------------
# 6. Store results
# --------------------------------------------------

# --------------------------------------------------
# 6. Store results
# --------------------------------------------------

final_results = pd.DataFrame(index=final_test.index)

if "Date" in final_test.columns:
    final_results["Date"] = final_test["Date"].values

final_results["actual"] = y_final_test.values
final_results["ensemble_pred"] = final_predictions

for col in final_base_predictions.columns:
    final_results[col] = final_base_predictions[col]

# --------------------------------------------------
# 7. Ensemble error scores
# --------------------------------------------------

y_true = final_results["actual"].values
y_pred = final_results["ensemble_pred"].values

mask = y_true != 0

ensemble_mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
ensemble_mae = mean_absolute_error(y_true, y_pred)
ensemble_rmse = np.sqrt(mean_squared_error(y_true, y_pred))

print("Ensemble MAE:", ensemble_mae)
print("Ensemble RMSE:", ensemble_rmse)
print("Ensemble MAPE:", ensemble_mape, "%")

final_results

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000265 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3787
[LightGBM] [Info] Number of data points in the train set: 1971, number of used features: 23
[LightGBM] [Info] Start training from score 21.781634
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive 

,actual,ensemble_pred,xgb_pred,lgb_pred,cat_pred
Date,,,,,
2023-11-10,17.7,18.942863,18.050892,18.217047,18.629842
2023-11-11,15.9,16.223907,15.742353,15.751412,15.890613
2023-11-12,11.7,12.771743,12.237761,12.374999,12.549218
2023-11-13,15.6,17.129904,15.881417,15.966599,16.380353
2023-11-14,18.3,18.704132,18.167067,18.128260,17.731458
2023-11-15,14.3,14.640573,14.125130,13.953347,14.128787
2023-11-16,17.6,18.704132,17.916815,17.854596,18.263026
2023-11-17,13.6,14.640573,14.226730,14.552994,14.700772
2023-11-18,17.6,17.771156,17.352413,17.391782,16.876377
